# Final RQ1 — frozen PhoBERT 27: validation and lock

Attach private code ZIP, intact 27-label results ZIP (or identical .bin), and authorized train/val CSVs. GPU + Internet. Run in a fresh session. This notebook ends before test inference; no retraining.


In [ ]:
from pathlib import Path
import os, shutil, zipfile
INPUT = Path('/kaggle/input')
ROOT = Path('/kaggle/working/ViGoEmotionsFinal')
if ROOT.exists():
    raise FileExistsError('Workspace exists; continue individual CLI stages instead of extracting over results.')
archives = list(INPUT.rglob('vigoemotions_final_code.zip'))
code_roots = [p.parent.parent for p in INPUT.rglob('scripts/final_thresholds.py')]
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        if any(not (ROOT / name).resolve().is_relative_to(ROOT.resolve()) for name in archive.namelist()):
            raise ValueError('Unsafe ZIP member')
        ROOT.mkdir(parents=True)
        archive.extractall(ROOT)
elif not archives and len(code_roots) == 1:
    shutil.copytree(code_roots[0], ROOT)
else:
    raise RuntimeError('Attach exactly one Final code dataset.')
os.chdir(ROOT)
print('Workspace:', ROOT)


In [ ]:
%pip install -q -r requirements-colab.txt


In [ ]:
import csv, subprocess, sys
if shutil.which('java') is None:
    raise RuntimeError('Install/enable Java before using the original VnCoreNLP segmenter.')
(ROOT / 'data').mkdir(exist_ok=True)
for split in ('train', 'val'):
    candidates = []
    for path in INPUT.rglob(split + '.csv'):
        with path.open(encoding='utf-8-sig', newline='') as stream:
            header = next(csv.reader(stream), [])
        if header == ['id', 'text', 'labels']:
            candidates.append(path)
    if len(candidates) != 1:
        raise RuntimeError('Attach exactly one authorized ' + split + '.csv source.')
    shutil.copy2(candidates[0], ROOT / 'data' / (split + '.csv'))
name = 'vigoemotions_kaggle_equal20_27labels_private_results'
weights = list(INPUT.rglob(name + '.zip')) + list(INPUT.rglob(name + '.bin'))
if len(weights) != 1:
    raise RuntimeError('Attach the intact private 27-label results ZIP or identical .bin, not only its extracted members.')
def run(stage, *extra):
    subprocess.run([sys.executable, 'scripts/final_thresholds.py', stage, *extra], check=True)
run('verify')
run('restore', '--archive', str(weights[0]))


In [ ]:
run('reproduce-val', '--device', 'cuda')


In [ ]:
run('fit')
import pandas as pd
display(pd.read_csv('reports/final/tables/val_comparison.csv'))
display(pd.read_csv('reports/final/tables/threshold_profile.csv'))


## Freeze after reviewing validation outputs

The next cell writes the pre-test lock. Keep this implementation/config unchanged afterward. Download reports/final and keep private assets available. Test inference is a separate later CLI action described in docs/final_execution.md.


In [ ]:
run('lock')
print('Validation complete and thresholds locked. No test inference performed.')
